<a href="https://colab.research.google.com/gist/FELIPEACASTRO/PLACEHOLDER_V14_GIST/kg1_v90_v14_colab_pro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KG1 v90 V14 — Colab Pro H100 — SURGICAL FIX (AdamW 32-bit + 9 LoRA targets)

**V14 corrige a OSCILAÇÃO de loss do V13** substituindo o optimizer (root cause identificada via double-check).

## V13 → V14 surgical diff:

| Componente | V13 (broken) | V14 (fix) | Fonte |
|---|---|---|---|
| Optimizer | `PagedAdam8bit` | **`AdamW 32-bit`** | qlora#78 (bug documentado) |
| max_grad_norm | (unset) | **1.0** | safety (dgxchen usa 1e9 = disabled) |
| LoRA targets | 8 (sem lm_head) | **9 (com lm_head)** | dgxchen LB 0.84 + konbu17 v2 |
| router aux_loss | (default) | **0.0 (disabled)** | ST-MoE z-loss instability |

## V14 PRESERVA tudo que V13 validou:
- NF4 + `skip_modules=['out_proj', 'lm_head']` (fix legendary V13)
- Custom training loop com balanced token normalization (Falcon-style)
- Stratified batching por subcategory (reduz variance)
- Data mix v90+v92+v93 (pipeline debugado)
- MAX_LENGTH=4096, eff_batch=32 (=dgxchen LB 0.84)
- Checkpoint save + Kaggle auto-submit a cada 100 steps
- Model soup (últimos 3 checkpoints)
- Mamba-ssm 2.3.1 + causal-conv1d 1.6.1 (torch 2.10 wheels)
- flex_attention OFF + triple list_repo_templates patch
- Colab keep-alive JavaScript

---

## Credenciais (Colab Secrets 🔑)

| Secret | Value |
|---|---|
| `HF_KEY` | seu token HF (valide em https://huggingface.co/settings/tokens) |
| `KAGGLE_USERNAME` | `felipe1983` |
| `KAGGLE_KEY` | `93dbcf741dba9085eded2cdbe2fc0cab` |

**Atenção**: HF tokens expiram. Se Cell 1 falhar com 'token invalid', gere novo em https://huggingface.co/settings/tokens (Expiration: No expiration).

---

## Expected Loss Trajectory V14 (target)

| Step | V13 (broken) | V14 (expected) |
|---|---|---|
| 10 | 0.58 | 0.60-0.80 |
| 25 | **1.82** spike | 0.45-0.60 |
| 45 | **1.16** spike | 0.30-0.45 |
| 50 | 0.68 (plateau) | 0.30-0.45 |
| 100 | — | 0.15-0.30 |
| 300 final | — | 0.10-0.20 |

## Kaggle LB prediction (90% CI from Claude-Opus-4.5 + DeepSeek):
- Floor: **0.78**
- Median: **0.82**
- Ceiling: **0.86** (match TOP 1)
- P(>= 0.75): 75% | P(>= 0.80): 55% | P(>= 0.84): 25%

## Execução

1. **Runtime → Change runtime → GPU → H100 High RAM**
2. Configure 3 secrets (HF_KEY, KAGGLE_USERNAME, KAGGLE_KEY)
3. **Runtime → Run all**
4. Aguarde 6-8h (até 4 Kaggle auto-submits)

Math VRAM V14: NF4 model (16 GB) + LoRA 9 targets (0.6) + AdamW32 (8) + activations (12) = **~37 GB / 80 GB**

**Safe headroom: 43 GB** (zero OOM risk)

## Cell 1 — Setup (GPU + secrets + HF token fail-fast)

In [ ]:
import os, sys, subprocess, json, time, math

!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
print('Python: ' + sys.version.split()[0])

ram = !free -g | head -2 | tail -1
disk = !df -h / | tail -1
print('RAM:', ' '.join(ram))
print('Disk:', ' '.join(disk))

from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_KEY')
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

assert os.environ.get('HF_TOKEN', '').startswith('hf_'), 'HF_KEY missing'
assert os.environ.get('KAGGLE_USERNAME'), 'KAGGLE_USERNAME missing'
assert os.environ.get('KAGGLE_KEY'), 'KAGGLE_KEY missing'
print('[OK] Secrets loaded')

from huggingface_hub import whoami
try:
    info = whoami(token=os.environ['HF_TOKEN'])
    print('[OK] HF Token valid — User: ' + info['name'])
except Exception as e:
    print('!!! HF TOKEN INVALID: ' + str(e)[:200])
    print('Fix: https://huggingface.co/settings/tokens (Expiration: No expiration)')
    print('Depois: atualize Colab secret HF_KEY + Runtime > Restart session')
    raise RuntimeError('HF token invalid')

import torch
gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print('')
print('GPU: ' + gpu_name)
print('VRAM: ' + str(round(gpu_mem_gb, 1)) + ' GB')
print('torch: ' + torch.__version__)

# V14: NF4 + skip_modules mantido (V13 proven)
USE_NF4 = True
MICRO_BATCH = 2
MAX_LENGTH = 4096
ABORT_VRAM_GIB = 77.0
print('[CONFIG] V14: NF4=True + skip_modules, MICRO=2, MAX_LEN=4096')
print('[OPTIMIZER V14 FIX] AdamW 32-bit (fixes V13 oscillation)')

# Colab keep-alive
try:
    from IPython.display import display, Javascript
    display(Javascript("setInterval(() => { "
        "document.querySelector('colab-toolbar-button#connect')?.click(); "
        "}, 60000);"))
    print('[OK] Keep-alive enabled (60s)')
except Exception:
    pass

## Cell 2 — GDrive mount + resume (V14)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_BASE = '/content/drive/MyDrive/KG1_v90_V14'
LOCAL_BASE = '/content/kg1'
CHECKPOINT_DIR = GDRIVE_BASE + '/checkpoints'
SUBMISSIONS_DIR = GDRIVE_BASE + '/submissions'
LOGS_DIR = GDRIVE_BASE + '/logs'
DATA_CACHE_DIR = GDRIVE_BASE + '/data_cache'

for d in [GDRIVE_BASE, CHECKPOINT_DIR, SUBMISSIONS_DIR, LOGS_DIR, DATA_CACHE_DIR, LOCAL_BASE]:
    os.makedirs(d, exist_ok=True)

print('[OK] GDrive: ' + GDRIVE_BASE)
gs = !df -h /content/drive | tail -1
print('Space:', ' '.join(gs))

import glob
existing = sorted(glob.glob(CHECKPOINT_DIR + '/checkpoint-*'),
                  key=lambda p: int(p.rsplit('checkpoint-', 1)[-1]) if 'checkpoint-' in p else -1)
RESUME_FROM_STEP = 0
if existing:
    RESUME_FROM_STEP = int(existing[-1].rsplit('checkpoint-', 1)[-1])
    print('[RESUME] step ' + str(RESUME_FROM_STEP))
else:
    print('[FRESH RUN V14]')

## Cell 3 — Install deps (torchao uninstall + mamba-ssm wheels)

In [ ]:
import subprocess

print('[1/4] Uninstall torchao (conflict peft>0.16)...')
subprocess.run(['pip', 'uninstall', '-y', 'torchao'], capture_output=True, text=True, timeout=60)

print('[2/4] Install core packages...')
!pip install -q \
    "transformers==4.55.0" \
    "tokenizers==0.21.0" \
    "huggingface_hub>=0.34,<1.0" \
    "peft>=0.14" \
    "bitsandbytes>=0.44" \
    "accelerate>=1.7" \
    "datasets>=3.0" \
    "safetensors>=0.5" \
    "kaggle>=1.6" \
    "ortools>=9.11" \
    "sympy>=1.13" \
    "trl>=0.12" \
    "protobuf>=4.25,<5.0" \
    --force-reinstall --no-deps 2>&1 | tail -3

print('[3/4] Install mamba-ssm 2.3.1 + causal-conv1d 1.6.1 (torch 2.10 wheels)...')
WHEELS = [
    'https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
    'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
]
for url in WHEELS:
    r = subprocess.run(['pip', 'install', '-q', url, '--no-deps'], capture_output=True, text=True, timeout=180)
    print('  [' + ('OK' if r.returncode == 0 else 'FAIL') + '] ' + url.split('/')[-1][:40])

print('[4/4] Clear cache + verify...')
for m in list(sys.modules.keys()):
    if any(m.startswith(p) for p in ['transformers', 'tokenizers', 'huggingface', 'peft', 'mamba_ssm', 'causal_conv1d']):
        del sys.modules[m]

import torch, transformers, tokenizers, peft, accelerate, huggingface_hub, bitsandbytes
import mamba_ssm, causal_conv1d
from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn

print('  torch:           ' + torch.__version__)
print('  transformers:    ' + transformers.__version__)
print('  tokenizers:      ' + tokenizers.__version__)
print('  huggingface_hub: ' + huggingface_hub.__version__)
print('  peft:            ' + peft.__version__)
print('  bitsandbytes:    ' + bitsandbytes.__version__)
print('  mamba_ssm:       ' + mamba_ssm.__version__)
print('  causal_conv1d:   ' + causal_conv1d.__version__)

assert transformers.__version__.startswith('4.55')
assert tokenizers.__version__.startswith('0.21')
assert not huggingface_hub.__version__.startswith('1.')

try:
    import torchao
    print('[WARN] torchao still present — may cause peft issues')
except ImportError:
    print('[OK] torchao removed')

print('[OK] V14 env ready')

## Cell 4 — Download scripts + data HF

In [ ]:
from huggingface_hub import hf_hub_download
SCRIPTS_REPO = 'felipesp1983/kg1-nemotron-training'
HF_TOKEN = os.environ['HF_TOKEN']

files = [
    'scripts/kg1_generate_synthetic_v92.py',
    'scripts/kg1_generate_augmentations_v93.py',
    'scripts/kg1_balanced_token_normalization.py',
    'scripts/kg1_stratified_batching.py',
    'scripts/kg1_model_soup.py',
    'scripts/kg1_calibration_manifest.py',
    'scripts/kg1_nemotron_adapter_converter.py',
    'scripts/kg1_submission_gate.py',
    'scripts/kg1_local_metric_gate.py',
    'src/perfect_solver.py',
    'src/teacher_cot.py',
    'src/competition_utils.py',
    'data/v90/v90_train_gold_safe.jsonl',
    'data/v90/v90_val_gold_safe_stratified.jsonl',
    'data/kaggle/unzipped/train.csv',
    'data/kaggle/unzipped/test.csv',
]

def dl(f, retries=3):
    lp = os.path.join(LOCAL_BASE, f)
    if os.path.exists(lp) and os.path.getsize(lp) > 0:
        return 'CACHED'
    for i in range(retries):
        try:
            hf_hub_download(repo_id=SCRIPTS_REPO, filename=f, repo_type='dataset',
                            local_dir=LOCAL_BASE, token=HF_TOKEN)
            return 'OK'
        except Exception as e:
            if i == retries - 1:
                return 'FAIL: ' + str(e)[:60]
            time.sleep(2 ** i)

missing = []
for f in files:
    s = dl(f)
    print('[' + s.split(':')[0] + '] ' + f)
    if 'FAIL' in s:
        missing.append(f)

if missing:
    raise RuntimeError(str(len(missing)) + ' files missing')

sys.path.insert(0, LOCAL_BASE)
sys.path.insert(0, LOCAL_BASE + '/scripts')
from kg1_stratified_batching import stratified_batch_order
from kg1_balanced_token_normalization import compute_balanced_loss, BalancedLossAccumulator
print('[OK] V14 helpers imported')

## Cell 5 — Kaggle CLI + daily limit

In [ ]:
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
kj = os.path.expanduser('~/.kaggle/kaggle.json')
with open(kj, 'w') as f:
    json.dump({'username': os.environ['KAGGLE_USERNAME'], 'key': os.environ['KAGGLE_KEY']}, f)
os.chmod(kj, 0o600)
print('[OK] kaggle.json')

r = subprocess.run(['kaggle', 'competitions', 'submissions',
                    '-c', 'nvidia-nemotron-model-reasoning-challenge', '--csv'],
                   capture_output=True, text=True, timeout=60)

from datetime import datetime, timezone
now = datetime.now(timezone.utc)
today = now.replace(hour=0, minute=0, second=0, microsecond=0)
count = 0
if r.returncode == 0:
    import csv, io
    for row in csv.DictReader(io.StringIO(r.stdout)):
        try:
            dt = datetime.strptime(row['date'], '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc)
            if dt >= today: count += 1
        except: pass

KAGGLE_SLOTS_REMAINING = max(0, 5 - count)
print('Submits today: ' + str(count) + '/5 | Slots remaining: ' + str(KAGGLE_SLOTS_REMAINING))

## Cell 6 — Generate v92+v93 (optional, timeout 2min)

In [ ]:
import multiprocessing as mp, shutil
nw = mp.cpu_count()
print('CPU cores: ' + str(nw))

v92g = DATA_CACHE_DIR + '/v92_combined.jsonl'
v92l = LOCAL_BASE + '/data/synthetic/v92/v92_combined.jsonl'
os.makedirs(os.path.dirname(v92l), exist_ok=True)

if os.path.exists(v92g):
    shutil.copy(v92g, v92l)
    print('[CACHED] v92')
else:
    print('Generating v92 (timeout 120s)...')
    try:
        r = subprocess.run(
            ['python', 'scripts/kg1_generate_synthetic_v92.py',
             '--output-dir', 'data/synthetic/v92', '--workers', str(nw)],
            cwd=LOCAL_BASE, capture_output=True, text=True, timeout=120)
        if r.returncode == 0 and os.path.exists(v92l):
            shutil.copy(v92l, v92g)
            print('[OK] v92 generated')
        else:
            print('[SKIP v92] ' + r.stderr[-200:])
            open(v92l, 'w').close()
    except subprocess.TimeoutExpired:
        print('[SKIP v92] timeout — using empty')
        open(v92l, 'w').close()

v93g = DATA_CACHE_DIR + '/v93_combined.jsonl'
v93l = LOCAL_BASE + '/data/augmentations/v93/v93_combined.jsonl'
os.makedirs(os.path.dirname(v93l), exist_ok=True)

if os.path.exists(v93g):
    shutil.copy(v93g, v93l)
    print('[CACHED] v93')
else:
    print('Generating v93 (timeout 120s)...')
    try:
        r = subprocess.run(
            ['python', 'scripts/kg1_generate_augmentations_v93.py',
             '--output-dir', 'data/augmentations/v93', '--total', '10000'],
            cwd=LOCAL_BASE, capture_output=True, text=True, timeout=120)
        if r.returncode == 0 and os.path.exists(v93l):
            shutil.copy(v93l, v93g)
            print('[OK] v93 generated')
        else:
            print('[SKIP v93] ' + r.stderr[-200:])
            open(v93l, 'w').close()
    except subprocess.TimeoutExpired:
        print('[SKIP v93] timeout — using empty')
        open(v93l, 'w').close()

def cl(p): return sum(1 for _ in open(p, 'r', encoding='utf-8')) if os.path.exists(p) else 0
print('v92: ' + str(cl(v92l)) + ' | v93: ' + str(cl(v93l)))

## Cell 7 — Build v91 mix (75/25)

In [ ]:
import random
def lj(p):
    if not os.path.exists(p) or os.path.getsize(p) == 0: return []
    return [json.loads(l) for l in open(p, 'r', encoding='utf-8') if l.strip()]

v90_train = lj(LOCAL_BASE + '/data/v90/v90_train_gold_safe.jsonl')
v90_val   = lj(LOCAL_BASE + '/data/v90/v90_val_gold_safe_stratified.jsonl')
v92 = lj(LOCAL_BASE + '/data/synthetic/v92/v92_combined.jsonl')
v93 = lj(LOCAL_BASE + '/data/augmentations/v93/v93_combined.jsonl')

def valid(r):
    ms = r.get('messages', [])
    return isinstance(ms, list) and len(ms) >= 2 and any(m.get('role') == 'assistant' for m in ms)

v90_train = [r for r in v90_train if valid(r)]
v92 = [r for r in v92 if valid(r)]
v93 = [r for r in v93 if valid(r)]
v90_val = [r for r in v90_val if valid(r)]

reasoning = v90_train + v92
max_aug = min(len(v93), int(len(reasoning) / 3)) if v93 else 0
rng = random.Random(42)
rng.shuffle(v93)
aug = v93[:max_aug]
mixed = reasoning + aug
rng.shuffle(mixed)
print('Mix total=' + str(len(mixed)) + ' reasoning=' + str(len(reasoning)) + ' aug=' + str(len(aug)))

## Cell 8 — Load Nemotron NF4 + skip_modules (V13 FIX PRESERVED)

In [ ]:
# V14 MANTÉM V13 fix: NF4 + skip_modules=['out_proj', 'lm_head']
# out_proj BF16 → Mamba fast_path funciona
# lm_head BF16 → LoRA pode ser attached sem OOM

# PATCH 1: flex_attention OFF (torch 2.10)
import transformers.utils
import transformers.utils.import_utils
transformers.utils.is_torch_flex_attn_available = lambda: False
transformers.utils.import_utils.is_torch_flex_attn_available = lambda: False
print('[PATCH] flex_attention disabled')

# PATCH 2: Triple-layer list_repo_templates
from huggingface_hub import HfApi
_orig = HfApi.list_repo_tree
def _safe(self, *a, **k):
    try: return list(_orig(self, *a, **k))
    except Exception as e:
        if '404' in str(e) or 'Not Found' in str(e): return []
        raise
HfApi.list_repo_tree = _safe
import transformers.utils.hub as tuh
tuh.list_repo_templates = lambda *a, **k: []
import transformers.tokenization_utils_base as tub
tub.list_repo_templates = lambda *a, **k: []
print('[PATCH] Triple-layer list_repo_templates')

# PATCH 3: expandable_segments
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=HF_TOKEN)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print('Tokenizer vocab: ' + str(len(tokenizer)))

# NF4 + skip_modules — V13 FIX MANTIDO
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_skip_modules=['out_proj', 'lm_head'],  # V13 FIX
)

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation='sdpa',
    quantization_config=bnb_cfg,
    token=HF_TOKEN,
)

# V14 FIX: Disable MoE router aux_loss (router is frozen, aux_loss causes z-loss instability)
if hasattr(model.config, 'output_router_logits'):
    model.config.output_router_logits = False
    print('[V14 FIX] output_router_logits = False')
if hasattr(model.config, 'router_aux_loss_coef'):
    model.config.router_aux_loss_coef = 0.0
    print('[V14 FIX] router_aux_loss_coef = 0.0')

print('[OK] Model loaded NF4+skip in ' + str(round((time.time()-t0)/60, 1)) + ' min')
print('VRAM: ' + str(round(torch.cuda.memory_allocated()/1e9, 1)) + ' GB (esperado ~16 GB)')
print('VRAM free: ' + str(round((torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9, 1)) + ' GB')

## Cell 9 — Apply LoRA V14 (9 targets COM lm_head, dgxchen LB 0.84 match)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# V14 FIX: 9 targets (add lm_head) — dgxchen LB 0.84 + konbu17 v2 LB 0.834 proven
# lm_head BF16 (via skip_modules) + LoRA attached → ~10 MB extra VRAM
# out_proj EXCLUÍDO (Mamba custom kernels access .weight directly, NVIDIA official rec)
TARGET = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj',
          'gate_proj', 'up_proj', 'down_proj', 'lm_head']  # V14: +lm_head

lc = LoraConfig(
    r=32, lora_alpha=32, lora_dropout=0.0,
    target_modules=TARGET,
    bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lc)

tr = [n for n, p in model.named_parameters() if p.requires_grad]
rt = [n for n in tr if any(k in n.lower() for k in ['router', 'gate_linear', 'expert_gate'])]
print('[' + ('WARN' if rt else 'OK') + '] router trainable: ' + str(len(rt)))
total = sum(p.numel() for p in model.parameters())
trnb = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Trainable: ' + format(trnb, ',') + ' (' + str(round(100*trnb/total, 2)) + '%)')
print('VRAM pós-LoRA: ' + str(round(torch.cuda.memory_allocated()/1e9, 1)) + ' GB')
print('VRAM free: ' + str(round((torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9, 1)) + ' GB')

## Cell 10 — Tokenize (MAX_LENGTH=4096)

In [ ]:
def build_mask(ms, tk):
    try:
        full = tk.apply_chat_template(ms, tokenize=False, add_generation_prompt=False, enable_thinking=True)
    except TypeError:
        full = tk.apply_chat_template(ms, tokenize=False, add_generation_prompt=False)
    fids = tk.encode(full, add_special_tokens=False)
    pm = [m for m in ms if m.get('role') != 'assistant']
    try:
        pt = tk.apply_chat_template(pm, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    except TypeError:
        pt = tk.apply_chat_template(pm, tokenize=False, add_generation_prompt=True)
    pids = tk.encode(pt, add_special_tokens=False)
    pl = min(len(pids), len(fids))
    mk = [0]*pl + [1]*(len(fids)-pl)
    if len(fids) > MAX_LENGTH:
        fids = fids[:MAX_LENGTH]; mk = mk[:MAX_LENGTH]
    return fids, mk

def tkds(records, label):
    out = []
    for r in records:
        try:
            ids, m = build_mask(r.get('messages', []), tokenizer)
            if sum(m) == 0: continue
            out.append({'input_ids': ids, 'loss_mask': m,
                        'subcategory': r.get('subcategory') or r.get('family', 'unknown')})
        except: pass
    print(label + ': ' + str(len(out)) + '/' + str(len(records)))
    return out

train_data = tkds(mixed, 'train')
val_data = tkds(v90_val, 'val')
assert len(train_data) > 100

## Cell 11 — Training V14 (AdamW 32-bit FIX + grad_clip 1.0 + 300 steps)

**V14 primary fix**: substitui `PagedAdam8bit` (qlora#78 bug) por `AdamW 32-bit`.
Adiciona `max_grad_norm=1.0` (safety vs dgxchen 1e9 disabled). ~+4 GB VRAM, safe.

In [ ]:
import zipfile, hashlib
LR = 2e-4; BATCH = 32; GRAD_ACCUM = BATCH // MICRO_BATCH
MAX_STEPS = 300; SAVE_EVERY = 100; EVAL_EVERY = 50
MAX_GRAD_NORM = 1.0  # V14 FIX: safety clip

if RESUME_FROM_STEP > 0:
    rd = CHECKPOINT_DIR + '/checkpoint-' + str(RESUME_FROM_STEP)
    if os.path.exists(rd + '/adapter_model.safetensors'):
        model.load_adapter(rd, adapter_name='default', is_trainable=True)
        print('[RESUMED] step ' + str(RESUME_FROM_STEP))

# V14 FIX: AdamW 32-bit (NOT PagedAdam8bit — qlora#78 documents oscillation bug)
opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.0,
)
print('Optimizer: AdamW 32-bit (V14 FIX)')

def lr_at(s): return LR * max(0.0, 1 - s/MAX_STEPS)

def collate(b):
    ml = max(len(x['input_ids']) for x in b)
    pi = tokenizer.pad_token_id
    i, a, l = [], [], []
    for x in b:
        n = len(x['input_ids'])
        i.append(x['input_ids'] + [pi]*(ml-n))
        a.append([1]*n + [0]*(ml-n))
        l.append(x['loss_mask'] + [0]*(ml-n))
    return {'input_ids': torch.tensor(i, dtype=torch.long).cuda(),
            'attention_mask': torch.tensor(a, dtype=torch.long).cuda(),
            'loss_mask': torch.tensor(l, dtype=torch.float32).cuda()}

submits_used = 0
def save_submit(step):
    global submits_used
    cl = LOCAL_BASE + '/ckpt_tmp'
    cg = CHECKPOINT_DIR + '/checkpoint-' + str(step)
    shutil.rmtree(cl, ignore_errors=True)
    model.save_pretrained(cl)
    tokenizer.save_pretrained(cl)
    shutil.copytree(cl, cg, dirs_exist_ok=True)
    print('  [SAVED]')
    ac = cl + '/adapter_config.json'; ab = cl + '/adapter_model.safetensors'
    if not (os.path.exists(ac) and os.path.exists(ab)): return None
    cfg = json.load(open(ac))
    # V14 gate: includes lm_head now
    if 'in_proj' not in cfg['target_modules']: return None
    if 'lm_head' not in cfg['target_modules']: return None
    if any(m in cfg['target_modules'] for m in ['x_proj', 'out_proj']): return None
    if cfg['r'] > 32: return None
    sz = SUBMISSIONS_DIR + '/v90v14-step' + str(step) + '.zip'
    with zipfile.ZipFile(sz, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(ac, arcname='adapter_config.json')
        z.write(ab, arcname='adapter_model.safetensors')
    with open(sz, 'rb') as f: sha = hashlib.sha256(f.read()).hexdigest()
    print('  [ZIP] ' + sha[:12])
    if submits_used >= KAGGLE_SLOTS_REMAINING:
        print('  [SKIP submit] daily slots exhausted'); return sha
    msg = 'v90v14 step' + str(step) + ' sha=' + sha[:12]
    r = subprocess.run(['kaggle', 'competitions', 'submit',
                        '-c', 'nvidia-nemotron-model-reasoning-challenge',
                        '-f', sz, '-m', msg],
                       capture_output=True, text=True, timeout=300)
    if r.returncode == 0:
        submits_used += 1
        print('  [KAGGLE OK] ' + str(submits_used) + '/' + str(KAGGLE_SLOTS_REMAINING))
    else:
        print('  [KAGGLE FAIL] ' + r.stderr[:200])
    with open(LOGS_DIR + '/submit-step' + str(step) + '.json', 'w') as f:
        json.dump({'step': step, 'sha': sha, 'rc': r.returncode, 'ts': time.time()}, f)
    return sha

model.train()
gs = RESUME_FROM_STEP
start = time.time()
abort = False
eval_losses = []
submit_shas = {}
data = stratified_batch_order(train_data, BATCH, 'subcategory', seed=42)

for ss in range(gs*BATCH, len(data), BATCH):
    if gs >= MAX_STEPS: break
    sb = data[ss:ss+BATCH]
    if len(sb) < BATCH: continue
    for pg in opt.param_groups: pg['lr'] = lr_at(gs)
    acc = BalancedLossAccumulator()
    opt.zero_grad()
    for mbs in range(0, len(sb), MICRO_BATCH):
        mb = collate(sb[mbs:mbs+MICRO_BATCH])
        out = model(input_ids=mb['input_ids'], attention_mask=mb['attention_mask'])
        ls, n = compute_balanced_loss(out.logits, mb['input_ids'], mb['loss_mask'], shift=True)
        acc.add(ls, n)
        mloss = ls / n.clamp(min=1.0)
        (mloss / GRAD_ACCUM).backward()
    # V14 FIX: gradient clipping before opt.step() (V13 didn't clip!)
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM
    )
    opt.step()
    gs += 1
    if gs % 50 == 0:
        torch.cuda.empty_cache()
    sl = acc.finalize().item()
    vr = torch.cuda.memory_reserved() / 1e9
    el = time.time() - start
    if gs % 5 == 0 or gs == RESUME_FROM_STEP + 1:
        print('step ' + str(gs) + '/' + str(MAX_STEPS) + ' loss ' + format(sl, '.4f')
              + ' vram ' + format(vr, '.1f') + ' | ' + format(el/60, '.1f') + 'm')
    if vr > ABORT_VRAM_GIB: abort = True; print('!!! ABORT VRAM'); break
    if math.isnan(sl) or math.isinf(sl): abort = True; print('!!! ABORT NaN'); break
    # V14: relaxed threshold (with grad clip, loss should NOT exceed 30 even at start)
    if gs >= 25 and sl > 2.0: abort = True; print('!!! ABORT loss > 2.0 at step ' + str(gs)); break
    if gs % EVAL_EVERY == 0:
        model.eval()
        es, en = 0.0, 0
        with torch.no_grad():
            for es_s in range(0, min(200, len(val_data)), MICRO_BATCH):
                emb = collate(val_data[es_s:es_s+MICRO_BATCH])
                eo = model(input_ids=emb['input_ids'], attention_mask=emb['attention_mask'])
                lss, nn = compute_balanced_loss(eo.logits, emb['input_ids'], emb['loss_mask'])
                es += lss.item(); en += nn.item()
        ell = es / max(1, en)
        eval_losses.append((gs, ell))
        print('  [eval] ' + str(gs) + ' loss ' + format(ell, '.4f'))
        if ell > 1.5 and gs <= 100: abort = True; print('!!! ABORT eval > 1.5'); break
        model.train()
    if gs % SAVE_EVERY == 0:
        print('=== Step ' + str(gs) + ' SAVE+SUBMIT ===')
        sh = save_submit(gs)
        if sh: submit_shas[gs] = sh

if not abort:
    model.save_pretrained(CHECKPOINT_DIR + '/final')
    tokenizer.save_pretrained(CHECKPOINT_DIR + '/final')
print('Training done. ' + str(gs) + ' steps in ' + format((time.time()-start)/60, '.1f') + 'm')
with open(LOGS_DIR + '/training_log.json', 'w') as f:
    json.dump({'steps': gs, 'elapsed_min': (time.time()-start)/60,
               'eval_losses': eval_losses, 'submit_shas': submit_shas, 'abort': abort}, f, indent=2)

## Cell 12 — Model soup (últimos 3 checkpoints)

In [ ]:
import glob
ckpts = sorted(glob.glob(CHECKPOINT_DIR + '/checkpoint-*'),
               key=lambda p: int(p.rsplit('checkpoint-', 1)[-1]))
print('Ckpts: ' + str([os.path.basename(c) for c in ckpts]))
if len(ckpts) >= 3:
    l3 = ckpts[-3:]
    sd = CHECKPOINT_DIR + '/soup'
    !cd {LOCAL_BASE} && python scripts/kg1_model_soup.py --checkpoints {' '.join(l3)} --output-dir {sd} --weights uniform 2>&1 | tail -15
    if os.path.exists(sd + '/adapter_model.safetensors'):
        szf = SUBMISSIONS_DIR + '/v90v14-soup.zip'
        with zipfile.ZipFile(szf, 'w', zipfile.ZIP_DEFLATED) as z:
            z.write(sd + '/adapter_config.json', arcname='adapter_config.json')
            z.write(sd + '/adapter_model.safetensors', arcname='adapter_model.safetensors')
        with open(szf, 'rb') as f: sh = hashlib.sha256(f.read()).hexdigest()
        print('[OK] Soup: ' + sh[:12])
        print('Submit: kaggle competitions submit -c nvidia-nemotron-model-reasoning-challenge -f ' + szf + ' -m "v14 soup"')

## Cell 13 — Upload HF + summary

In [ ]:
from huggingface_hub import HfApi, create_repo
best = CHECKPOINT_DIR + '/soup' if os.path.exists(CHECKPOINT_DIR + '/soup/adapter_model.safetensors') else CHECKPOINT_DIR + '/final'
HF_REPO = 'felipesp1983/kg1-nemotron-lora-v90-v14'
api = HfApi(token=HF_TOKEN)
try:
    create_repo(HF_REPO, private=True, exist_ok=True, token=HF_TOKEN)
    api.upload_folder(folder_path=best, repo_id=HF_REPO, repo_type='model')
    print('[OK] Uploaded -> ' + HF_REPO)
except Exception as e:
    print('[WARN] ' + str(e)[:200])

print('')
print('=== V14 Submits ===')
for s, sh in sorted(submit_shas.items()):
    print('  step ' + str(s) + ': ' + sh[:16])
print('')
print('LB: https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/submissions')